###### =========================
#### 📊 Data Analytics - Silver to Gold Layer
###### =========================

###### This notebook transforms cleaned data from the Silver layer
###### into business-level analytics and insights.

#### Objectives:
###### - Compute global KPIs
###### - Analyze revenue and activity trends
###### - Identify top products and customers
###### - Perform RFM customer segmentation

### 📊 Global KPIs

This section provides a high-level overview of business performance using key metrics.

We calculate:
- Total Revenue
- Total Orders
- Total Customers
- Average Order Value

###### Load Silver data

In [12]:
df = spark.read.table("silver.online_retail")
display(df.head(5))

StatementMeta(, b06828db-ee6e-4b2d-bacc-3b513109f600, 27, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 98b8a618-8d1e-4526-af7d-486639df2e5a)

##### 📥 Silver Data Validation

###### Observations:
- Data is successfully loaded from the Silver layer
- Records appear clean, structured, and enriched with engineered features:
  - `line_total` (revenue per transaction line)
  - `is_return` (return indicator)
  - `year`, `month` (time dimensions)  

- Sample shows valid transactional data:
  - Positive quantities and prices  
  - Consistent timestamp format  
  - Presence of customer and country information  

###### Interpretation:
- Data cleaning and transformation steps were successfully applied in the Silver layer  
- Dataset is ready for analytical processing in the Gold layer  

This confirms that the Silver dataset is reliable and suitable for business analytics.

###### KPIs

In [2]:
from pyspark.sql.functions import *

kpis = df.agg(
    round(sum("line_total"),2).alias("Total_Revenue"),
    countDistinct("InvoiceNo").alias("Total_Orders"),
    countDistinct("CustomerID").alias("Total_Customers")
)

StatementMeta(, b06828db-ee6e-4b2d-bacc-3b513109f600, 5, Finished, Available, Finished, False)

###### Add Average Order Value

In [3]:
kpis = kpis.withColumn(
    "Avg_Order_Value",
    round(kpis["Total_Revenue"] / kpis["Total_Orders"],2)
)

display(kpis)

StatementMeta(, b06828db-ee6e-4b2d-bacc-3b513109f600, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, bb20f12a-6e05-403b-bbe7-24d24cb46758)

##### 📊 Global KPIs Analysis

###### Observations:
- Total Revenue: **8.3M** → indicates strong overall sales performance  
- Total Orders: **22,190** → reflects a high transaction volume  
- Total Customers: **4,372** → shows a relatively broad customer base  
- Average Order Value: **373.07** → suggests high spending per order  

###### Interpretation:
- The business generates significant revenue with a moderate number of customers  
- The relatively high average order value indicates that customers tend to purchase multiple items or higher-value products per transaction  

###### Business Insight:
- Focus on customer retention could further increase revenue, given the strong spending behavior  
- High AOV suggests opportunities for upselling and bundled offers  

These KPIs confirm a healthy and revenue-generating e-commerce activity.

### 📈 Revenue & Activity Trends

This section analyzes how business performance evolves over time.

We focus on:
- Monthly revenue
- Number of orders
- Number of customers

In [4]:
monthly_trends = df.groupBy("year", "month") \
    .agg(
        round(sum("line_total"),2).alias("monthly_revenue"),
        countDistinct("InvoiceNo").alias("monthly_orders"),
        countDistinct("CustomerID").alias("monthly_customers")
    ) \
    .orderBy("year", "month")

display(monthly_trends)

StatementMeta(, b06828db-ee6e-4b2d-bacc-3b513109f600, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e8c13436-3d00-4de5-bbbf-aa758f0e257e)

##### 📈 Trend Analysis

###### Observations:
- Revenue shows an overall increasing trend throughout 2011  
- Significant growth from September to November, peaking at **1.13M in November**  
- December 2011 shows a sharp drop (**342K**), likely due to incomplete data or seasonality  

- Orders and customers follow a similar pattern:
  - Peak activity in **November (3086 orders, 1711 customers)**  
  - Strong growth starting from September  

###### Interpretation:
- The business experiences strong seasonality with peak performance in Q4 (especially November)  
- The alignment between revenue, orders, and customers indicates consistent demand growth rather than price-driven spikes  

###### Business Insight:
- Q4 (Sep–Nov) is the most critical period → ideal for marketing campaigns and inventory planning  
- November appears to be the highest-performing month → potential impact of holiday sales (e.g., Black Friday)  
- The drop in December should be interpreted cautiously (possible partial data)

These trends highlight clear seasonal patterns and growth opportunities for the business.


### 🏆 Top Products & Customers

This section identifies the best-performing products and most valuable customers.

We analyze:
- Top products by revenue
- Top products by quantity  
- Top customers by total spending

###### 🔹 Top Products by revenue

In [5]:
top_products = df.groupBy("StockCode", "Description") \
    .agg(round(sum("line_total"),2).alias("total_revenue")) \
    .orderBy("total_revenue", ascending=False)

display(top_products.limit(10))

StatementMeta(, b06828db-ee6e-4b2d-bacc-3b513109f600, 8, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 34c7f416-cfd3-43ab-b5eb-6969f3fea75c)

##### 🏆 Top Products Analysis

###### Observations:
- The top product is **REGENCY CAKESTAND 3 TIER** with revenue of **132K**, significantly ahead of others  
- Several decorative and gift-related items dominate the ranking:
  - Home decor (e.g., *T-LIGHT HOLDER*, *BIRD ORNAMENT*)
  - Seasonal or event products (e.g., *CHRISTMAS items*, *PARTY BUNTING*)  

- The presence of **POSTAGE** in top revenue indicates a notable contribution from shipping fees  

###### Interpretation:
- The business is strongly driven by **gift-oriented and decorative products**  
- High-performing items suggest a focus on **events, holidays, and home decoration**  
- Revenue concentration on a few products highlights key drivers of sales  

###### Business Insight:
- Top products should be prioritized for inventory management and promotions  
- Seasonal items (e.g., Christmas products) confirm the importance of peak periods  
- Shipping costs (POSTAGE) should be monitored as they represent a non-product revenue stream  

This analysis highlights the main revenue drivers and supports strategic product and marketing decisions.


🔹 Top Products by quantity

In [6]:
top_products_qty = df.groupBy("StockCode", "Description") \
    .agg(sum("Quantity").alias("total_quantity")) \
    .orderBy("total_quantity", ascending=False)

display(top_products_qty.limit(10))

StatementMeta(, b06828db-ee6e-4b2d-bacc-3b513109f600, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 7e59af2a-87a2-431c-a361-d3d4a12cb37d)

##### 📦 Top Products by Quantity

###### Observations:
- The most sold product is **WORLD WAR 2 GLIDERS** with **53K units**, far ahead of others  
- Several low-price, high-volume items dominate:
  - Packaging or small items (e.g., *CAKE CASES*, *TISSUES*, *POPCORN HOLDER*)  
  - Decorative but affordable products  

- Some products appear in both rankings (quantity & revenue):
  - *JUMBO BAG RED RETROSPOT*  
  - *BIRD ORNAMENT*  
  - *T-LIGHT HOLDER*  

###### Interpretation:
- High-volume products are typically **low-cost items sold in bulk**  
- There is a clear distinction between:
  - **High-revenue products** (premium / higher price)  
  - **High-volume products** (low price / frequently purchased)  

- Products appearing in both lists are **key business drivers** (high volume + strong revenue)  

###### Business Insight:
- High-volume items are critical for maintaining steady sales flow  
- These products are ideal for promotions, bundles, or cross-selling strategies  
- Identifying products that perform well in both revenue and quantity is key for maximizing profitability  

This analysis provides a deeper understanding of product performance beyond revenue alone.


🔹 Top Customers

In [7]:
top_customers = df.groupBy("CustomerID") \
    .agg(round(sum("line_total"),2).alias("total_spent")) \
    .orderBy("total_spent", ascending=False)

display(top_customers.limit(10))

StatementMeta(, b06828db-ee6e-4b2d-bacc-3b513109f600, 10, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 0c91fc9a-7be0-4f7a-8d76-ef5e7a212f32)

##### 👥 Top Customers Analysis

###### Observations:
- The top customer (**ID 14646**) generated **~279K**, significantly higher than others  
- The top 3 customers each contributed over **180K**, showing strong individual impact  
- There is a noticeable gap between top customers and the rest of the list  

###### Interpretation:
- Revenue is partially concentrated among a small group of high-value customers  
- These customers likely represent **key accounts or highly loyal buyers**  
- The distribution suggests a typical **Pareto effect (80/20 rule)** where a minority drives a large share of revenue  

###### Business Insight:
- Top customers should be prioritized for retention (loyalty programs, personalized offers)  
- Losing a small number of these clients could significantly impact revenue  
- Opportunity to identify and nurture similar high-value customers  

This analysis highlights the strategic importance of high-spending customers in overall business performance.

#### 💎 RFM Segmentation

This section segments customers based on their purchasing behavior using RFM analysis:

- Recency: How recently a customer made a purchase  
- Frequency: How often a customer makes purchases  
- Monetary: How much a customer spends  

#### 🔢 RFM Scoring

Customers are ranked into 5 groups (quintiles) for each metric:
- Scores range from 1 to 5  
- Higher scores indicate more valuable customers  

This enables effective customer segmentation and targeting. 

###### RFM calculation

In [8]:
# Reference date = today
reference_date = current_date()

rfm = df.groupBy("CustomerID").agg(
    datediff(reference_date, max("InvoiceDate")).alias("Recency"),
    countDistinct("InvoiceNo").alias("Frequency"),
    round(sum("line_total"),2).alias("Monetary")
)

display(rfm)

StatementMeta(, b06828db-ee6e-4b2d-bacc-3b513109f600, 11, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 46d7b44c-0b8c-4933-80f6-4cf18e97c477)

##### 💎 RFM Analysis

###### Observations:
- Recency values are relatively high (around 5200–5300 days), indicating that many customers have not purchased recently  
- Frequency varies significantly:
  - Some customers have very low frequency (1–3 orders)  
  - Others show strong engagement (e.g., 54 orders)  

- Monetary values also vary widely:
  - Low spenders (<200)  
  - High-value customers (>10K and even >30K)  
  - ⚠️ Several customers have **negative Monetary values**, indicating returns or refunds exceeding purchases  

###### Interpretation:
- The dataset contains a mix of:
  - Inactive or one-time customers  
  - Highly engaged and high-spending customers  

- Customers with high Frequency and Monetary (e.g., Customer 16013, 16333) represent **key business assets**  
- High Recency suggests many customers are no longer active → potential churn  
- Negative Monetary values highlight customer returns behavior, which can distort analysis  

###### Business Insight:
- Focus should be placed on re-engaging inactive customers (high Recency)  
- High Frequency & Monetary customers should be targeted for retention and loyalty programs  
- It is important to **handle or filter negative Monetary values** before further segmentation  
- Customer segmentation is essential to differentiate marketing strategies  

This analysis highlights customer diversity and the importance of targeted engagement strategies.


#### 💾 Gold Layer Storage

The final dataset containing KPIs, trends, top analysis, and RFM segmentation is saved in the Gold layer.

This layer is optimized for business analytics and reporting.

In [10]:
kpis.write.mode("overwrite").saveAsTable("gold.kpis")

monthly_trends.write.mode("overwrite").saveAsTable("gold.monthly_trends")

top_products.write.mode("overwrite").saveAsTable("gold.top_revenue")

top_products_qty.write.mode("overwrite").saveAsTable("gold.top_quantity")

top_customers.write.mode("overwrite").saveAsTable("gold.top_customers")

rfm.write.mode("overwrite").saveAsTable("gold.rfm")


print("✅ Data successfully written to Gold layer")

StatementMeta(, b06828db-ee6e-4b2d-bacc-3b513109f600, 13, Finished, Available, Finished, False)

✅ Data successfully written to Gold layer


##### 🥇 Gold Layer (Business Analytics)

The Gold layer contains curated, business-ready datasets designed for analytics and reporting.

###### 📊 Available Datasets:

- **KPIs**
  - Global business metrics (revenue, orders, customers, AOV)

- **Trends**
  - Monthly revenue, orders, and customer evolution

- **Products**
  - Top products by revenue
  - Top products by quantity

- **Customers**
  - Top customers by total spending
  - RFM segmentation (Recency, Frequency, Monetary)

###### 🎯 Purpose:
These datasets are optimized for:
- Business intelligence dashboards  
- Data visualization tools (Power BI, Tableau)  
- Decision-making and strategic analysis  

##### ⚠️ Note:
Data in this layer is cleaned, aggregated, and structured for direct consumption.